In [3]:
# =========================================
# LOAD FINAL ML DATASET
# =========================================

import pandas as pd
import numpy as np

DATA_PATH = "data/cleaned/final_ml_dataset.csv"

df = pd.read_csv(DATA_PATH)

# Convert time column
df["Data Acquisition Time"] = pd.to_datetime(df["Data Acquisition Time"])

# Ensure chronological order
df = df.sort_values("Data Acquisition Time").reset_index(drop=True)

print("=" * 70)
print("FINAL ML DATASET LOADED")
print("=" * 70)

print(f"Shape: {df.shape}")
print(
    f"Date range: "
    f"{df['Data Acquisition Time'].min()} → "
    f"{df['Data Acquisition Time'].max()}"
)
print(f"Target column: High_Water_6h")

FINAL ML DATASET LOADED
Shape: (7309, 21)
Date range: 2022-01-04 04:00:00 → 2026-06-03 17:00:00
Target column: High_Water_6h


In [4]:
# =========================================
# TIME-BASED TRAIN / TEST SPLIT
# =========================================

TARGET = "High_Water_6h"
TIME_COL = "Data Acquisition Time"

# 80% earlier observations for training
# 20% later observations for testing
split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

print("=" * 70)
print("TIME-BASED SPLIT")
print("=" * 70)

print(f"Total rows : {len(df)}")
print(f"Train rows : {len(train_df)}")
print(f"Test rows  : {len(test_df)}")

print("\nTraining period:")
print(f"{train_df[TIME_COL].min()} → {train_df[TIME_COL].max()}")

print("\nTesting period:")
print(f"{test_df[TIME_COL].min()} → {test_df[TIME_COL].max()}")

print("\nChronological boundary check:")

if train_df[TIME_COL].max() < test_df[TIME_COL].min():
    print("PASS — Training data occurs entirely before test data.")
else:
    print("FAIL — Temporal overlap detected.")

TIME-BASED SPLIT
Total rows : 7309
Train rows : 5847
Test rows  : 1462

Training period:
2022-01-04 04:00:00 → 2025-04-06 03:00:00

Testing period:
2025-04-06 04:00:00 → 2026-06-03 17:00:00

Chronological boundary check:
PASS — Training data occurs entirely before test data.


In [5]:
# =========================================
# SEPARATE FEATURES AND TARGET
# =========================================

TARGET = "High_Water_6h"
TIME_COL = "Data Acquisition Time"

# Features used by the ML models
FEATURES = [
    "Current_Water_Level",
    "Month",
    "Hour",
    "DayOfYear",
    "WaterLevel_Lag_1h",
    "WaterLevel_Lag_3h",
    "WaterLevel_Lag_6h",
    "WaterLevel_Lag_12h",
    "WaterLevel_Lag_24h",
    "WaterLevel_RollingMean_6h",
    "WaterLevel_RollingMean_12h",
    "WaterLevel_RollingMean_24h",
    "WaterLevel_RollingMax_24h",
    "WaterLevel_RollingMin_24h",
    "WaterLevel_Change_1h",
    "WaterLevel_Change_3h",
    "WaterLevel_Change_6h",
    "WaterLevel_Change_12h",
    "WaterLevel_Change_24h"
]

X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].copy()

X_test = test_df[FEATURES].copy()
y_test = test_df[TARGET].copy()

print("=" * 70)
print("FEATURE / TARGET SEPARATION")
print("=" * 70)

print(f"Number of features : {len(FEATURES)}")
print(f"X_train shape      : {X_train.shape}")
print(f"y_train shape      : {y_train.shape}")
print(f"X_test shape       : {X_test.shape}")
print(f"y_test shape       : {y_test.shape}")

print("\nTarget distribution — Training:")
print(y_train.value_counts().sort_index())

print("\nTarget distribution — Testing:")
print(y_test.value_counts().sort_index())

FEATURE / TARGET SEPARATION
Number of features : 19
X_train shape      : (5847, 19)
y_train shape      : (5847,)
X_test shape       : (1462, 19)
y_test shape       : (1462,)

Target distribution — Training:
High_Water_6h
0.0    5580
1.0     267
Name: count, dtype: int64

Target distribution — Testing:
High_Water_6h
0.0     265
1.0    1197
Name: count, dtype: int64


In [6]:
# =========================================
# TARGET DISTRIBUTION OVER TIME
# =========================================

print("=" * 70)
print("TARGET DISTRIBUTION OVER TIME")
print("=" * 70)

# Monthly target distribution
monthly_target = (
    df.set_index(TIME_COL)[TARGET]
      .resample("ME")
      .agg(["count", "sum", "mean"])
)

monthly_target["positive_rate_%"] = monthly_target["mean"] * 100

print(monthly_target.to_string())

TARGET DISTRIBUTION OVER TIME
                       count    sum      mean  positive_rate_%
Data Acquisition Time                                         
2022-01-31                66    0.0  0.000000         0.000000
2022-02-28                74    0.0  0.000000         0.000000
2022-03-31                82    0.0  0.000000         0.000000
2022-04-30                84    0.0  0.000000         0.000000
2022-05-31                82    0.0  0.000000         0.000000
2022-06-30                96    0.0  0.000000         0.000000
2022-07-31                89    0.0  0.000000         0.000000
2022-08-31                75    0.0  0.000000         0.000000
2022-09-30                74    0.0  0.000000         0.000000
2022-10-31                66    0.0  0.000000         0.000000
2022-11-30                77    0.0  0.000000         0.000000
2022-12-31                68    0.0  0.000000         0.000000
2023-01-31               179    0.0  0.000000         0.000000
2023-02-28               

In [7]:
# =========================================
# TIME-BASED VALIDATION SPLIT
# =========================================

VALIDATION_SPLIT = 0.20

validation_split_index = int(len(train_df) * (1 - VALIDATION_SPLIT))

model_train_df = train_df.iloc[:validation_split_index].copy()
validation_df = train_df.iloc[validation_split_index:].copy()

print("=" * 70)
print("TIME-BASED VALIDATION SPLIT")
print("=" * 70)

print(f"Original training rows : {len(train_df)}")
print(f"Model-training rows    : {len(model_train_df)}")
print(f"Validation rows        : {len(validation_df)}")

print("\nModel-training period:")
print(
    f"{model_train_df[TIME_COL].min()} → "
    f"{model_train_df[TIME_COL].max()}"
)

print("\nValidation period:")
print(
    f"{validation_df[TIME_COL].min()} → "
    f"{validation_df[TIME_COL].max()}"
)

print("\nChronological boundary check:")

if model_train_df[TIME_COL].max() < validation_df[TIME_COL].min():
    print("PASS — Model-training data occurs entirely before validation data.")
else:
    print("FAIL — Temporal overlap detected.")

TIME-BASED VALIDATION SPLIT
Original training rows : 5847
Model-training rows    : 4677
Validation rows        : 1170

Model-training period:
2022-01-04 04:00:00 → 2024-09-05 01:00:00

Validation period:
2024-09-05 02:00:00 → 2025-04-06 03:00:00

Chronological boundary check:
PASS — Model-training data occurs entirely before validation data.


In [8]:
# =========================================
# PREPARE MODEL TRAINING / VALIDATION DATA
# =========================================

# Model-training data
X_model_train = model_train_df[FEATURES].copy()
y_model_train = model_train_df[TARGET].copy()

# Validation data
X_validation = validation_df[FEATURES].copy()
y_validation = validation_df[TARGET].copy()

print("=" * 70)
print("MODEL TRAINING / VALIDATION DATA")
print("=" * 70)

print(f"X_model_train shape : {X_model_train.shape}")
print(f"y_model_train shape : {y_model_train.shape}")

print(f"X_validation shape  : {X_validation.shape}")
print(f"y_validation shape  : {y_validation.shape}")

print("\nTarget distribution — Model Training:")
print(y_model_train.value_counts().sort_index())

print("\nTarget distribution — Validation:")
print(y_validation.value_counts().sort_index())

MODEL TRAINING / VALIDATION DATA
X_model_train shape : (4677, 19)
y_model_train shape : (4677,)
X_validation shape  : (1170, 19)
y_validation shape  : (1170,)

Target distribution — Model Training:
High_Water_6h
0.0    4677
Name: count, dtype: int64

Target distribution — Validation:
High_Water_6h
0.0    903
1.0    267
Name: count, dtype: int64


In [9]:
# =========================================
# FIND FIRST POSITIVE TARGET EVENT
# =========================================

positive_rows = df[df[TARGET] == 1].copy()

print("=" * 70)
print("FIRST POSITIVE TARGET EVENT")
print("=" * 70)

print(f"Total positive events: {len(positive_rows)}")

print(f"\nFirst positive event:")
print(positive_rows[TIME_COL].min())

print(f"\nLast positive event:")
print(positive_rows[TIME_COL].max())

FIRST POSITIVE TARGET EVENT
Total positive events: 1464

First positive event:
2025-01-04 18:00:00

Last positive event:
2026-06-03 17:00:00


In [10]:
# =========================================
# REVISED TIME-BASED VALIDATION SPLIT
# =========================================

VALIDATION_START = pd.Timestamp("2025-02-01 00:00:00")

model_train_df = train_df[
    train_df[TIME_COL] < VALIDATION_START
].copy()

validation_df = train_df[
    train_df[TIME_COL] >= VALIDATION_START
].copy()

print("=" * 70)
print("REVISED TIME-BASED VALIDATION SPLIT")
print("=" * 70)

print(f"Development/Training period rows : {len(train_df)}")
print(f"Model-training rows              : {len(model_train_df)}")
print(f"Validation rows                  : {len(validation_df)}")

print("\nModel-training period:")
print(
    f"{model_train_df[TIME_COL].min()} → "
    f"{model_train_df[TIME_COL].max()}"
)

print("\nValidation period:")
print(
    f"{validation_df[TIME_COL].min()} → "
    f"{validation_df[TIME_COL].max()}"
)

print("\nTarget distribution — Model Training:")
print(
    model_train_df[TARGET]
    .value_counts()
    .sort_index()
)

print("\nTarget distribution — Validation:")
print(
    validation_df[TARGET]
    .value_counts()
    .sort_index()
)

print("\nChronological boundary check:")

if model_train_df[TIME_COL].max() < validation_df[TIME_COL].min():
    print("PASS — Model-training data occurs entirely before validation data.")
else:
    print("FAIL — Temporal overlap detected.")

REVISED TIME-BASED VALIDATION SPLIT
Development/Training period rows : 5847
Model-training rows              : 5561
Validation rows                  : 286

Model-training period:
2022-01-04 04:00:00 → 2025-01-08 17:00:00

Validation period:
2025-02-02 00:00:00 → 2025-04-06 03:00:00

Target distribution — Model Training:
High_Water_6h
0.0    5478
1.0      83
Name: count, dtype: int64

Target distribution — Validation:
High_Water_6h
0.0    102
1.0    184
Name: count, dtype: int64

Chronological boundary check:
PASS — Model-training data occurs entirely before validation data.


In [14]:
# =========================================
# PREPARE REVISED MODEL TRAINING / VALIDATION DATA
# =========================================

X_model_train = model_train_df[FEATURES].copy()
y_model_train = model_train_df[TARGET].copy()

X_validation = validation_df[FEATURES].copy()
y_validation = validation_df[TARGET].copy()

print("=" * 70)
print("REVISED MODEL TRAINING / VALIDATION DATA")
print("=" * 70)

print(f"X_model_train shape : {X_model_train.shape}")
print(f"y_model_train shape : {y_model_train.shape}")

print(f"X_validation shape  : {X_validation.shape}")
print(f"y_validation shape  : {y_validation.shape}")

print("\nTarget distribution — Model Training:")
print(y_model_train.value_counts().sort_index())

print("\nTarget distribution — Validation:")
print(y_validation.value_counts().sort_index())

REVISED MODEL TRAINING / VALIDATION DATA
X_model_train shape : (5561, 19)
y_model_train shape : (5561,)
X_validation shape  : (286, 19)
y_validation shape  : (286,)

Target distribution — Model Training:
High_Water_6h
0.0    5478
1.0      83
Name: count, dtype: int64

Target distribution — Validation:
High_Water_6h
0.0    102
1.0    184
Name: count, dtype: int64


In [15]:
# =========================================
# LOGISTIC REGRESSION — BASELINE MODEL
# =========================================

from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

# Train the model
logistic_model.fit(
    X_model_train,
    y_model_train
)

print("=" * 70)
print("LOGISTIC REGRESSION TRAINING")
print("=" * 70)

print("Model training completed successfully.")
print(f"Classes learned: {logistic_model.classes_}")
print(f"Number of features learned: {logistic_model.coef_.shape[1]}")

LOGISTIC REGRESSION TRAINING
Model training completed successfully.
Classes learned: [0. 1.]
Number of features learned: 19


In [16]:
# =========================================
# LOGISTIC REGRESSION — VALIDATION PREDICTION
# =========================================

y_validation_pred = logistic_model.predict(X_validation)

y_validation_proba = logistic_model.predict_proba(X_validation)[:, 1]

print("=" * 70)
print("VALIDATION PREDICTION")
print("=" * 70)

print(f"Validation samples       : {len(X_validation)}")
print(f"Predictions generated    : {len(y_validation_pred)}")
print(f"Probability values       : {len(y_validation_proba)}")

print("\nFirst 10 predictions:")
print(y_validation_pred[:10])

print("\nFirst 10 class-1 probabilities:")
print(y_validation_proba[:10])

VALIDATION PREDICTION
Validation samples       : 286
Predictions generated    : 286
Probability values       : 286

First 10 predictions:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

First 10 class-1 probabilities:
[4.33325114e-14 7.90613792e-14 1.44249698e-13 8.76125840e-13
 2.91653734e-12 5.32130395e-12 1.77141011e-11 3.23198729e-11
 5.89685121e-11 1.07589700e-10]


In [17]:
# =========================================
# LOGISTIC REGRESSION — VALIDATION EVALUATION
# =========================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

accuracy = accuracy_score(y_validation, y_validation_pred)
precision = precision_score(
    y_validation,
    y_validation_pred,
    zero_division=0
)
recall = recall_score(
    y_validation,
    y_validation_pred,
    zero_division=0
)
f1 = f1_score(
    y_validation,
    y_validation_pred,
    zero_division=0
)

cm = confusion_matrix(
    y_validation,
    y_validation_pred
)

print("=" * 70)
print("LOGISTIC REGRESSION — VALIDATION RESULTS")
print("=" * 70)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_validation,
        y_validation_pred,
        zero_division=0
    )
)

LOGISTIC REGRESSION — VALIDATION RESULTS
Accuracy  : 0.3636
Precision : 1.0000
Recall    : 0.0109
F1-Score  : 0.0215

Confusion Matrix:
[[102   0]
 [182   2]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.36      1.00      0.53       102
         1.0       1.00      0.01      0.02       184

    accuracy                           0.36       286
   macro avg       0.68      0.51      0.28       286
weighted avg       0.77      0.36      0.20       286



In [18]:
# =========================================
# SCALED LOGISTIC REGRESSION — BASELINE
# =========================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaled_logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

scaled_logistic_model.fit(
    X_model_train,
    y_model_train
)

print("=" * 70)
print("SCALED LOGISTIC REGRESSION TRAINING")
print("=" * 70)

print("Training completed successfully.")

SCALED LOGISTIC REGRESSION TRAINING
Training completed successfully.


In [19]:
# =========================================
# SCALED LOGISTIC REGRESSION — VALIDATION
# =========================================

y_validation_pred_lr = scaled_logistic_model.predict(
    X_validation
)

print("=" * 70)
print("SCALED LOGISTIC REGRESSION — VALIDATION PREDICTION")
print("=" * 70)

print(f"Validation predictions: {len(y_validation_pred_lr)}")

print("\nPredicted class distribution:")
print(
    pd.Series(y_validation_pred_lr)
    .value_counts()
    .sort_index()
)

SCALED LOGISTIC REGRESSION — VALIDATION PREDICTION
Validation predictions: 286

Predicted class distribution:
0.0    107
1.0    179
Name: count, dtype: int64


In [20]:
# =========================================
# SCALED LOGISTIC REGRESSION — EVALUATION
# =========================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

accuracy_lr = accuracy_score(
    y_validation,
    y_validation_pred_lr
)

precision_lr = precision_score(
    y_validation,
    y_validation_pred_lr,
    zero_division=0
)

recall_lr = recall_score(
    y_validation,
    y_validation_pred_lr,
    zero_division=0
)

f1_lr = f1_score(
    y_validation,
    y_validation_pred_lr,
    zero_division=0
)

cm_lr = confusion_matrix(
    y_validation,
    y_validation_pred_lr
)

print("=" * 70)
print("SCALED LOGISTIC REGRESSION — VALIDATION RESULTS")
print("=" * 70)

print(f"Accuracy  : {accuracy_lr:.4f}")
print(f"Precision : {precision_lr:.4f}")
print(f"Recall    : {recall_lr:.4f}")
print(f"F1-Score  : {f1_lr:.4f}")

print("\nConfusion Matrix:")
print(cm_lr)

print("\nClassification Report:")
print(
    classification_report(
        y_validation,
        y_validation_pred_lr,
        zero_division=0
    )
)

SCALED LOGISTIC REGRESSION — VALIDATION RESULTS
Accuracy  : 0.9266
Precision : 0.9553
Recall    : 0.9293
F1-Score  : 0.9421

Confusion Matrix:
[[ 94   8]
 [ 13 171]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.88      0.92      0.90       102
         1.0       0.96      0.93      0.94       184

    accuracy                           0.93       286
   macro avg       0.92      0.93      0.92       286
weighted avg       0.93      0.93      0.93       286



In [21]:
# =========================================
# DECISION TREE — BASELINE MODEL
# =========================================

from sklearn.tree import DecisionTreeClassifier

decision_tree_model = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42
)

decision_tree_model.fit(
    X_model_train,
    y_model_train
)

print("=" * 70)
print("DECISION TREE TRAINING")
print("=" * 70)

print("Training completed successfully.")

DECISION TREE TRAINING
Training completed successfully.


In [22]:
# =========================================
# DECISION TREE — VALIDATION PREDICTION
# =========================================

y_validation_pred_dt = decision_tree_model.predict(
    X_validation
)

print("=" * 70)
print("DECISION TREE — VALIDATION PREDICTION")
print("=" * 70)

print(f"Validation predictions: {len(y_validation_pred_dt)}")

print("\nPredicted class distribution:")
print(
    pd.Series(y_validation_pred_dt)
    .value_counts()
    .sort_index()
)

DECISION TREE — VALIDATION PREDICTION
Validation predictions: 286

Predicted class distribution:
0.0    102
1.0    184
Name: count, dtype: int64


In [23]:
# =========================================
# DECISION TREE — VALIDATION EVALUATION
# =========================================

accuracy_dt = accuracy_score(
    y_validation,
    y_validation_pred_dt
)

precision_dt = precision_score(
    y_validation,
    y_validation_pred_dt,
    zero_division=0
)

recall_dt = recall_score(
    y_validation,
    y_validation_pred_dt,
    zero_division=0
)

f1_dt = f1_score(
    y_validation,
    y_validation_pred_dt,
    zero_division=0
)

cm_dt = confusion_matrix(
    y_validation,
    y_validation_pred_dt
)

print("=" * 70)
print("DECISION TREE — VALIDATION RESULTS")
print("=" * 70)

print(f"Accuracy  : {accuracy_dt:.4f}")
print(f"Precision : {precision_dt:.4f}")
print(f"Recall    : {recall_dt:.4f}")
print(f"F1-Score  : {f1_dt:.4f}")

print("\nConfusion Matrix:")
print(cm_dt)

print("\nClassification Report:")
print(
    classification_report(
        y_validation,
        y_validation_pred_dt,
        zero_division=0
    )
)

DECISION TREE — VALIDATION RESULTS
Accuracy  : 0.9510
Precision : 0.9620
Recall    : 0.9620
F1-Score  : 0.9620

Confusion Matrix:
[[ 95   7]
 [  7 177]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.93      0.93      0.93       102
         1.0       0.96      0.96      0.96       184

    accuracy                           0.95       286
   macro avg       0.95      0.95      0.95       286
weighted avg       0.95      0.95      0.95       286



In [24]:
# =========================================
# RANDOM FOREST — BASELINE MODEL
# =========================================

from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(
    X_model_train,
    y_model_train
)

print("=" * 70)
print("RANDOM FOREST TRAINING")
print("=" * 70)

print("Training completed successfully.")
print(f"Number of trees: {random_forest_model.n_estimators}")

RANDOM FOREST TRAINING
Training completed successfully.
Number of trees: 200


In [25]:
# =========================================
# RANDOM FOREST — VALIDATION PREDICTION
# =========================================

y_validation_pred_rf = random_forest_model.predict(
    X_validation
)

print("=" * 70)
print("RANDOM FOREST — VALIDATION PREDICTION")
print("=" * 70)

print(f"Validation predictions: {len(y_validation_pred_rf)}")

print("\nPredicted class distribution:")
print(
    pd.Series(y_validation_pred_rf)
    .value_counts()
    .sort_index()
)


RANDOM FOREST — VALIDATION PREDICTION
Validation predictions: 286

Predicted class distribution:
0.0    109
1.0    177
Name: count, dtype: int64


In [26]:
# =========================================
# RANDOM FOREST — VALIDATION EVALUATION
# =========================================

accuracy_rf = accuracy_score(
    y_validation,
    y_validation_pred_rf
)

precision_rf = precision_score(
    y_validation,
    y_validation_pred_rf,
    zero_division=0
)

recall_rf = recall_score(
    y_validation,
    y_validation_pred_rf,
    zero_division=0
)

f1_rf = f1_score(
    y_validation,
    y_validation_pred_rf,
    zero_division=0
)

cm_rf = confusion_matrix(
    y_validation,
    y_validation_pred_rf
)

print("=" * 70)
print("RANDOM FOREST — VALIDATION RESULTS")
print("=" * 70)

print(f"Accuracy  : {accuracy_rf:.4f}")
print(f"Precision : {precision_rf:.4f}")
print(f"Recall    : {recall_rf:.4f}")
print(f"F1-Score  : {f1_rf:.4f}")

print("\nConfusion Matrix:")
print(cm_rf)

print("\nClassification Report:")
print(
    classification_report(
        y_validation,
        y_validation_pred_rf,
        zero_division=0
    )
)

RANDOM FOREST — VALIDATION RESULTS
Accuracy  : 0.9755
Precision : 1.0000
Recall    : 0.9620
F1-Score  : 0.9806

Confusion Matrix:
[[102   0]
 [  7 177]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.94      1.00      0.97       102
         1.0       1.00      0.96      0.98       184

    accuracy                           0.98       286
   macro avg       0.97      0.98      0.97       286
weighted avg       0.98      0.98      0.98       286



In [27]:
# =========================================
# GRADIENT BOOSTING — BASELINE MODEL
# =========================================

from sklearn.ensemble import GradientBoostingClassifier

gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gradient_boosting_model.fit(
    X_model_train,
    y_model_train
)

print("=" * 70)
print("GRADIENT BOOSTING TRAINING")
print("=" * 70)

print("Training completed successfully.")
print(f"Number of boosting stages: {gradient_boosting_model.n_estimators}")

GRADIENT BOOSTING TRAINING
Training completed successfully.
Number of boosting stages: 100


In [28]:
# =========================================
# GRADIENT BOOSTING — VALIDATION PREDICTION
# =========================================

y_validation_pred_gb = gradient_boosting_model.predict(
    X_validation
)

print("=" * 70)
print("GRADIENT BOOSTING — VALIDATION PREDICTION")
print("=" * 70)

print(f"Validation predictions: {len(y_validation_pred_gb)}")

print("\nPredicted class distribution:")
print(
    pd.Series(y_validation_pred_gb)
    .value_counts()
    .sort_index()
)

GRADIENT BOOSTING — VALIDATION PREDICTION
Validation predictions: 286

Predicted class distribution:
0.0    102
1.0    184
Name: count, dtype: int64


In [29]:
# =========================================
# GRADIENT BOOSTING — VALIDATION EVALUATION
# =========================================

accuracy_gb = accuracy_score(
    y_validation,
    y_validation_pred_gb
)

precision_gb = precision_score(
    y_validation,
    y_validation_pred_gb,
    zero_division=0
)

recall_gb = recall_score(
    y_validation,
    y_validation_pred_gb,
    zero_division=0
)

f1_gb = f1_score(
    y_validation,
    y_validation_pred_gb,
    zero_division=0
)

cm_gb = confusion_matrix(
    y_validation,
    y_validation_pred_gb
)

print("=" * 70)
print("GRADIENT BOOSTING — VALIDATION RESULTS")
print("=" * 70)

print(f"Accuracy  : {accuracy_gb:.4f}")
print(f"Precision : {precision_gb:.4f}")
print(f"Recall    : {recall_gb:.4f}")
print(f"F1-Score  : {f1_gb:.4f}")

print("\nConfusion Matrix:")
print(cm_gb)

print("\nClassification Report:")
print(
    classification_report(
        y_validation,
        y_validation_pred_gb,
        zero_division=0
    )
)

GRADIENT BOOSTING — VALIDATION RESULTS
Accuracy  : 0.9510
Precision : 0.9620
Recall    : 0.9620
F1-Score  : 0.9620

Confusion Matrix:
[[ 95   7]
 [  7 177]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.93      0.93      0.93       102
         1.0       0.96      0.96      0.96       184

    accuracy                           0.95       286
   macro avg       0.95      0.95      0.95       286
weighted avg       0.95      0.95      0.95       286



In [30]:
# =========================================
# SELECTED MODEL — FINAL DEVELOPMENT TRAINING
# =========================================

from sklearn.ensemble import RandomForestClassifier

final_random_forest = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

final_random_forest.fit(
    X_train,
    y_train
)

print("=" * 70)
print("FINAL RANDOM FOREST — DEVELOPMENT TRAINING")
print("=" * 70)

print(f"Training rows used : {len(X_train)}")
print(f"Features used      : {X_train.shape[1]}")
print(f"Number of trees    : {final_random_forest.n_estimators}")
print("Model training completed successfully.")

FINAL RANDOM FOREST — DEVELOPMENT TRAINING
Training rows used : 5847
Features used      : 19
Number of trees    : 200
Model training completed successfully.


In [31]:
# =========================================
# FINAL RANDOM FOREST — TEST PREDICTION
# =========================================

y_test_pred = final_random_forest.predict(
    X_test
)

print("=" * 70)
print("FINAL RANDOM FOREST — TEST PREDICTION")
print("=" * 70)

print(f"Test samples        : {len(X_test)}")
print(f"Predictions generated: {len(y_test_pred)}")

print("\nPredicted class distribution:")
print(
    pd.Series(y_test_pred)
    .value_counts()
    .sort_index()
)

FINAL RANDOM FOREST — TEST PREDICTION
Test samples        : 1462
Predictions generated: 1462

Predicted class distribution:
0.0     246
1.0    1216
Name: count, dtype: int64


In [32]:
# =========================================
# FINAL RANDOM FOREST — TEST EVALUATION
# =========================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

accuracy_final = accuracy_score(
    y_test,
    y_test_pred
)

precision_final = precision_score(
    y_test,
    y_test_pred,
    zero_division=0
)

recall_final = recall_score(
    y_test,
    y_test_pred,
    zero_division=0
)

f1_final = f1_score(
    y_test,
    y_test_pred,
    zero_division=0
)

cm_final = confusion_matrix(
    y_test,
    y_test_pred
)

print("=" * 70)
print("FINAL RANDOM FOREST — TEST RESULTS")
print("=" * 70)

print(f"Accuracy  : {accuracy_final:.4f}")
print(f"Precision : {precision_final:.4f}")
print(f"Recall    : {recall_final:.4f}")
print(f"F1-Score  : {f1_final:.4f}")

print("\nConfusion Matrix:")
print(cm_final)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_test_pred,
        zero_division=0
    )
)

FINAL RANDOM FOREST — TEST RESULTS
Accuracy  : 0.9856
Precision : 0.9836
Recall    : 0.9992
F1-Score  : 0.9913

Confusion Matrix:
[[ 245   20]
 [   1 1196]]

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      0.92      0.96       265
         1.0       0.98      1.00      0.99      1197

    accuracy                           0.99      1462
   macro avg       0.99      0.96      0.98      1462
weighted avg       0.99      0.99      0.99      1462



In [33]:
# =========================================
# SAVE FINAL RANDOM FOREST MODEL
# =========================================

import os
import joblib

MODEL_DIR = "models"

os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "floodsense_random_forest.joblib"
)

joblib.dump(
    final_random_forest,
    MODEL_PATH
)

print("=" * 70)
print("FINAL MODEL SAVED")
print("=" * 70)

print(f"Model path : {MODEL_PATH}")
print("Model type : Random Forest")
print(f"Trees      : {final_random_forest.n_estimators}")
print("Save status: SUCCESS")

FINAL MODEL SAVED
Model path : models\floodsense_random_forest.joblib
Model type : Random Forest
Trees      : 200
Save status: SUCCESS


In [34]:
# =========================================
# VERIFY SAVED FINAL MODEL
# =========================================

import joblib
import os

MODEL_PATH = "models/floodsense_random_forest.joblib"

# Reload saved model
loaded_model = joblib.load(MODEL_PATH)

print("=" * 70)
print("SAVED MODEL RELOAD VERIFICATION")
print("=" * 70)

print(f"Model file exists : {os.path.exists(MODEL_PATH)}")
print(f"Model type        : {type(loaded_model).__name__}")
print(f"Number of trees   : {loaded_model.n_estimators}")
print(f"Features expected : {loaded_model.n_features_in_}")

print("\nReload status: SUCCESS")

SAVED MODEL RELOAD VERIFICATION
Model file exists : True
Model type        : RandomForestClassifier
Number of trees   : 200
Features expected : 19

Reload status: SUCCESS
